# 03 · Index & retrieve
Step 3: embed the chunks into Chroma, build the bm25s keyword index, and test dense / BM25 / hybrid retrieval.

Section 1 calls the OpenAI embeddings API (~1.2M tokens for both strategies, under $0.03). Sections 2–4 only read the saved indexes (queries are embedded, fractions of a cent).

Restart the kernel after editing anything in `src/`.

In [8]:
import sys, pathlib
print(sys.executable)
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import json, pandas as pd
from IPython.display import Markdown, display
from finsight.index.vectorstore import build_all, MANIFEST, check_manifest
from finsight.index.bm25 import build_bm25
from finsight.retrieve.hybrid import search

c:\Users\senpr\Documents\finsight-ai\.venv\Scripts\python.exe


## 1. Build the indexes
Chroma collections `fixed` and `semantic` are rebuilt into an empty `index/chroma_db/` every run (in-place rebuilds leave orphaned data and bloat the folder). Vectors come from the embedding cache, so only changed chunk text calls the API. bm25s indexes go to `index/bm25/`.

**Restart the kernel before running this cell**, so no Chroma client holds the folder open.

In [9]:
counts = build_all()  # fresh index/chroma_db/ every time; vectors come from the embedding cache
for strategy in ['fixed', 'semantic']:
    build_bm25(strategy)
print('vectors:', counts)
check_manifest()
print(MANIFEST.read_text())

PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\senpr\\Documents\\finsight-ai\\index\\chroma_db\\303d210c-bd61-4622-81b7-e320343225be\\data_level0.bin'

In [ ]:
questions = [
    ('What was AWS operating income in 2024?', 'Amazon', 2024),
    ('How much revenue did Google Cloud generate in 2024?', 'Alphabet', 2024),
    ('What was Intelligent Cloud revenue in fiscal year 2024?', 'Microsoft', 2024),
    ('What was free cash flow in 2024?', 'Amazon', 2024),
    ('What are the main risks related to AI regulation?', 'Alphabet', 2025),
    ('How many employees does Microsoft have?', 'Microsoft', 2025),
]

def show(q, company=None, year=None, strategy='fixed', k=5):
    rows = []
    for mode in ['dense', 'bm25', 'hybrid']:
        for rank, d in enumerate(search(q, strategy, mode, company=company, fiscal_year=year, k=k), 1):
            m = d.metadata
            rows.append({'mode': mode, 'rank': rank, 'chunk_id': m['chunk_id'], 'kind': m['kind'],
                         'section': m['section'][:40], 'page': m['page'] or '-', 'title': m['title'][:50]})
    print(f'Q: {q}   [{company} {year}, {strategy}]')
    display(pd.DataFrame(rows).pivot(index='rank', columns='mode', values='chunk_id'))
    return pd.DataFrame(rows)

res = show(*questions[0])

Q: What was AWS operating income in 2024?   [Amazon 2024, fixed]


mode,bm25,dense,hybrid
rank,,,
1,amazon_2024-fixed-00084,amazon_2024-fixed-00084,amazon_2024-fixed-00084
2,amazon_2024-table-0004,amazon_2024-table-0004,amazon_2024-table-0004
3,amazon_2024-fixed-00151,amazon_2024-fixed-00000,amazon_2024-fixed-00000
4,amazon_2024-fixed-00000,amazon_2024-fixed-00079,amazon_2024-table-0031
5,amazon_2024-table-0031,amazon_2024-table-0031,amazon_2024-fixed-00083


In [ ]:
res[res['mode'] == 'hybrid']

,mode,rank,chunk_id,kind,section,page,title
10,hybrid,1,amazon_2024-fixed-00084,text,Item 7. Management’s Discussion and Anal,39,
11,hybrid,2,amazon_2024-table-0004,table,Item 7. Management’s Discussion and Anal,39,Operating income (loss) by segment is as follo...
12,hybrid,3,amazon_2024-fixed-00000,text,General,1,
13,hybrid,4,amazon_2024-table-0031,table,Note 7 — COMMITMENTS AND CONTINGENCIES,78,Information on reportable segments and reconci...
14,hybrid,5,amazon_2024-fixed-00083,text,Item 7. Management’s Discussion and Anal,38,


### Read the top hybrid hit
This is the text the LLM would see.

In [ ]:
top = search(questions[0][0], 'fixed', 'hybrid', company=questions[0][1], fiscal_year=questions[0][2], k=1)[0]
print(top.metadata)
display(Markdown(top.page_content[:3000]))

{'chunk_id': 'amazon_2024-fixed-00084', 'doc_id': 'amazon_2024', 'company': 'Amazon', 'fiscal_year': 2024, 'doc_type': 'annual_report', 'kind': 'text', 'section': 'Item 7. Management’s Discussion and Analysis of Financial Condition and Results of Operations', 'page': 39, 'title': ''}


Amazon | FY2024 | annual_report | Item 7. Management’s Discussion and Analysis of Financial Condition and Results of Operations | p.39
Operating income was $36.9 billion and $68.6 billion for 2023 and 2024. We believe that operating income is a more meaningful measure than gross profit and gross margin due to the diversity of our product categories and services. For more information on the operating expenses that impact segment operating income, see “Operating Expenses” and the descriptions of operating expense line item changes on pages 25 to 27, and “Note 10 — Segment Information” on page 65.
The increase in North America operating income in 2024, compared to the prior year, is primarily due to increased unit sales and increased advertising sales, partially offset by increased fulfillment and shipping costs.
The International operating income in 2024, as compared to the operating loss in the prior year, is primarily due to increased unit sales and increased advertising sales, partially offset by increased shipping and fulfillment costs. Changes in foreign exchange rates did not significantly impact operating income in 2024.
The increase in AWS operating income in 2024, compared to the prior year, is primarily due to increased sales, decreased payroll and related expenses, and a reduction in depreciation and amortization expense from our change in the estimated useful lives of our servers, partially offset by spending on technology infrastructure that was primarily driven by additional investments to support AWS business growth. Changes in foreign exchange rates positively impacted operating income by $240 million in 2024.
Interest Income and Expense Our interest income was $2.9 billion and $4.7 billion during 2023 and 2024, primarily due to a higher average balance of invested funds at prevailing rates. We generally invest our excess cash in investment grade short- to intermediate-term marketable debt securities and AAA-rated money market funds. Our interest income corresponds with the average balance of invested funds based on the prevailing rates, which vary depending on the geographies and currencies in which they are invested.
Interest expense was $3.2 billion and $2.4 billion in 2023 and 2024 and was primarily related to debt and finance leases.
See Item 8 of Part II, “Financial Statements and Supplementary Data — Note 4 — Leases and Note 6 — Debt” for additional information.

## 3. All test questions (hybrid, fixed)

In [ ]:
for q, company, year in questions[1:]:
    r = show(q, company, year)
    top = search(q, 'fixed', 'hybrid', company=company, fiscal_year=year, k=1)[0]
    print('TOP:', top.metadata['chunk_id'], '|', top.page_content[:300].replace(chr(10), ' '))
    print()

Q: How much revenue did Google Cloud generate in 2024?   [Alphabet 2024, fixed]


mode,bm25,dense,hybrid
rank,,,
1,alphabet_2024-fixed-00090,alphabet_2024-fixed-00073,alphabet_2024-table-0056
2,alphabet_2024-fixed-00172,alphabet_2024-table-0002,alphabet_2024-fixed-00073
3,alphabet_2024-table-0056,alphabet_2024-fixed-00113,alphabet_2024-table-0002
4,alphabet_2024-fixed-00112,alphabet_2024-table-0019,alphabet_2024-fixed-00080
5,alphabet_2024-fixed-00126,alphabet_2024-fixed-00079,alphabet_2024-fixed-00113


TOP: alphabet_2024-table-0056 | Alphabet | FY2024 | 10-K | Notes To Consolidated Financial Statements | p.89 Table: The following table presents revenue, profitability, and expense information about our segments (in millions): Units: in millions |  | 2022 | 2023 | 2024 | |---|---|---|---| | Revenues: |  |  |  | | Google Services |

Q: What was Intelligent Cloud revenue in fiscal year 2024?   [Microsoft 2024, fixed]


mode,bm25,dense,hybrid
rank,,,
1,microsoft_2024-fixed-00052,microsoft_2024-fixed-00032,microsoft_2024-fixed-00056
2,microsoft_2024-fixed-00055,microsoft_2024-fixed-00056,microsoft_2024-fixed-00055
3,microsoft_2024-fixed-00056,microsoft_2024-table-0066,microsoft_2024-table-0066
4,microsoft_2024-table-0066,microsoft_2024-fixed-00080,microsoft_2024-fixed-00052
5,microsoft_2024-fixed-00053,microsoft_2024-fixed-00055,microsoft_2024-fixed-00053


TOP: microsoft_2024-fixed-00056 | Microsoft | FY2024 | annual_report | Segment Results Of Operations Reportable Segments Fiscal Year 2024 Compared with Fiscal Year 2023 Productivity and Business Processes Revenue increased $8.5 billion or 12%. • Office Commercial products and cloud services revenue increased $5.8 billion or 14%. Off

Q: What was free cash flow in 2024?   [Amazon 2024, fixed]


...

TOP: amazon_2024-fixed-00087 | Amazon | FY2024 | annual_report | Item 7. Management’s Discussion and Analysis of Financial Condition and Results of Operations | p.40 Free Cash Flow Less Principal Repayments of Finance Leases and Financing Obligations Free cash flow less principal repayments of finance leases and financing obligat

Q: What are the main risks related to AI regulation?   [Alphabet 2025, fixed]


mode,bm25,dense,hybrid
rank,,,
1,alphabet_2025-fixed-00071,alphabet_2025-fixed-00042,alphabet_2025-fixed-00042
2,alphabet_2025-fixed-00047,alphabet_2025-fixed-00033,alphabet_2025-fixed-00032
3,alphabet_2025-fixed-00042,alphabet_2025-fixed-00043,alphabet_2025-fixed-00047
4,alphabet_2025-fixed-00018,alphabet_2025-fixed-00032,alphabet_2025-fixed-00036
5,alphabet_2025-fixed-00032,alphabet_2025-fixed-00036,alphabet_2025-fixed-00018


TOP: alphabet_2025-fixed-00042 | Alphabet | FY2025 | 10-K | Item 1A. Risk Factors | p.18 • AI: Laws and regulations focused on the development, use, and provision of AI technologies and other digital products and services, which could result in monetary penalties or other regulatory actions. For example, the EU AI Act came into for

Q: How many employees does Microsoft have?   [Microsoft 2025, fixed]


mode,bm25,dense,hybrid
rank,,,
1,microsoft_2025-fixed-00026,microsoft_2025-fixed-00026,microsoft_2025-fixed-00026
2,microsoft_2025-fixed-00107,microsoft_2025-fixed-00015,microsoft_2025-fixed-00030
3,microsoft_2025-fixed-00008,microsoft_2025-fixed-00037,microsoft_2025-fixed-00037
4,microsoft_2025-fixed-00001,microsoft_2025-fixed-00030,microsoft_2025-fixed-00107
5,microsoft_2025-fixed-00109,microsoft_2025-table-0061,microsoft_2025-fixed-00031


TOP: microsoft_2025-fixed-00026 | Microsoft | FY2025 | annual_report | Human Capital Resources As of June 30, 2025, we employed approximately 228,000 people on a full-time basis, 125,000 in the U.S. and 103,000 internationally. Of the total employees, 89,000 were in operations, including product support and consulting services, data



## 4. Same question, fixed vs semantic

In [ ]:
for strategy in ['fixed', 'semantic']:
    hits = search(questions[4][0], strategy, 'hybrid', company=questions[4][1], fiscal_year=questions[4][2], k=3)
    print(strategy)
    for d in hits:
        print('   ', d.metadata['chunk_id'], d.metadata['section'][:50], '|', d.page_content.split(chr(10), 1)[1][:150].replace(chr(10), ' '))

fixed
    alphabet_2025-fixed-00042 Item 1A. Risk Factors | • AI: Laws and regulations focused on the development, use, and provision of AI technologies and other digital products and services, which could resu
    alphabet_2025-fixed-00032 Item 1A. Risk Factors | Table of Contents Alphabet Inc. leases, backstops, guarantees, and potential liabilities from large commercial agreements, could harm our financial co
    alphabet_2025-fixed-00047 Item 1A. Risk Factors | For additional information about the ongoing material legal proceedings to which we are subject, see Legal Proceedings in Item 3 of this Annual Report
semantic
    alphabet_2025-semantic-00035 Item 1A. Risk Factors | • Competition and technology platforms' business practices: Laws and regulations focused on large technology platforms, including the Digital Markets 
    alphabet_2025-semantic-00028 Item 1A. Risk Factors | • different employee/employer relationships; different labor laws, regulations, and labor practices; and 

## Notes / issues found
_(fill in after reviewing)_